# Ariel Data Challenge 2025

In [ ]:
import numpy as np
import pandas as pd
import itertools
import os
import glob
import warnings
import gc

from scipy.stats import skew, kurtosis
from scipy.optimize import minimize
from scipy.special import softmax
from scipy.signal import savgol_filter
from astropy.stats import sigma_clip

import lightgbm as lgb
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from tqdm import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

CFG = dict(
    data_path   = '/kaggle/input/ariel-data-challenge-2025/',
    output_path = '/kaggle/working/',

    gain        = 0.4369,
    offset      = -1000.0,
    cut_inf     = 39,
    cut_sup     = 321,
    airs_bin    = 30,
    fgs1_bin    = 360,
    do_mask     = True,
    do_nl_corr  = False,
    do_dark     = True,
    do_flat     = True,

    n_wl             = 283,
    n_time           = 187,
    out_transit_frac = 0.15,

    n_folds  = 5,
    cv_seed  = 42,

    lgbm_params = dict(
        objective        = 'regression',
        metric           = 'rmse',
        n_estimators     = 3000,
        learning_rate    = 0.03,
        num_leaves       = 127,
        max_depth        = -1,
        min_child_samples= 20,
        subsample        = 0.8,
        subsample_freq   = 1,
        colsample_bytree = 0.8,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        n_jobs           = -1,
        random_state     = 42,
        verbose          = -1,
        device           = 'gpu',
    ),

    lgbm_q_extra = dict(
        objective    = 'quantile',
        n_estimators = 2000,
        learning_rate= 0.03,
    ),
    q_lo = 0.16,
    q_hi = 0.84,

    et_params = dict(
        n_estimators    = 300,
        max_features    = 0.6,
        min_samples_leaf= 5,
        n_jobs          = -1,
        random_state    = 123,
    ),

    sigma_floor        = 1e-5,
    smooth_predictions = False,
    smooth_window      = 7,
    smooth_poly        = 2,
)

PATH    = CFG['data_path']
OUT     = CFG['output_path']
N_WL    = CFG['n_wl']
N_TIME  = CFG['n_time']
cut_inf = CFG['cut_inf']
cut_sup = CFG['cut_sup']

## Calibration

In [ ]:
def ADC_convert(signal, gain=CFG['gain'], offset=CFG['offset']):
    """Invert detector ADC quantisation."""
    signal = signal.astype(np.float32)
    signal /= gain
    signal += offset
    return signal


def mask_hot_dead(signal, dead, dark):
    """Flag hot pixels (5-sigma dark clip) and dead pixels as masked."""
    hot  = np.tile(sigma_clip(dark, sigma=5, maxiters=5).mask, (signal.shape[0], 1, 1))
    dead = np.tile(dead, (signal.shape[0], 1, 1))
    return np.ma.masked_where(hot | dead, signal)


def apply_linear_corr(linear_corr, signal):
    """Per-pixel polynomial non-linearity correction."""
    linear_corr = np.flip(linear_corr, axis=0)
    for x, y in itertools.product(range(signal.shape[1]), range(signal.shape[2])):
        signal[:, x, y] = np.poly1d(linear_corr[:, x, y])(signal[:, x, y])
    return signal


def clean_dark(signal, dead, dark, dt):
    """Subtract scaled dark frame from signal."""
    dark = np.tile(np.ma.masked_where(dead, dark), (signal.shape[0], 1, 1))
    signal -= dark * dt[:, np.newaxis, np.newaxis]
    return signal


def get_cds(signal):
    """Correlated double sampling: difference of consecutive read pairs."""
    return signal[:, 1::2] - signal[:, ::2]


def bin_obs(signal, binning):
    """Sum consecutive time bins to reduce cadence."""
    n = signal.shape[1] // binning
    out = np.zeros((signal.shape[0], n, signal.shape[2], signal.shape[3]), dtype=np.float32)
    for i in range(n):
        out[:, i] = signal[:, i*binning:(i+1)*binning].sum(axis=1)
    return out


def correct_flat_field(flat, dead, signal):
    """Normalise by pixel response (flat field)."""
    flat = np.tile(np.ma.masked_where(dead.T, flat.T), (signal.shape[0], 1, 1))
    return signal / flat


def _load_cal(base, instrument, shape, crop=None):
    """Load calibration files for one instrument into a dict of arrays."""
    p = os.path.join(base, f'{instrument}_calibration_0')
    def _rd(name, s):
        arr = pd.read_parquet(os.path.join(p, f'{name}.parquet')).values.astype(np.float32).reshape(s)
        return arr[..., crop[0]:crop[1]] if crop else arr
    return {
        'dark':        _rd('dark',        shape),
        'dead':        _rd('dead',        shape).astype(bool),
        'flat':        _rd('flat',        shape),
        'linear_corr': _rd('linear_corr', (6, *shape)),
    }


def calibrate_planet(planet_id, split='train'):
    """
    Run the full calibration chain for a single planet observation.

    Steps: ADC inversion → hot/dead masking → dark subtraction →
    correlated double sampling → time binning → flat field correction.

    Returns
    -------
    airs_cal : ndarray (187, 282, 32)  float32
    fgs1_cal : ndarray (187,  32, 32)  float32
    """
    base      = os.path.join(PATH, split, str(planet_id))
    axis_info = pd.read_parquet(os.path.join(PATH, 'axis_info.parquet'))

    def _process(instrument, raw_shape, cal_shape, binning, dt, crop=None):
        df  = pd.read_parquet(os.path.join(base, f'{instrument}_signal_0.parquet'))
        sig = df.values.astype(np.float32).reshape((df.shape[0], *raw_shape))
        del df
        sig = ADC_convert(sig)
        if crop:
            sig = sig[:, :, crop[0]:crop[1]]

        cal = _load_cal(base, instrument, cal_shape, crop)

        if CFG['do_mask']:
            sig = mask_hot_dead(sig, cal['dead'], cal['dark'])
        if CFG['do_nl_corr']:
            sig = apply_linear_corr(cal['linear_corr'], np.ma.getdata(sig))
        if CFG['do_dark']:
            sig = clean_dark(sig, cal['dead'], cal['dark'], dt)

        sig4d  = np.ma.getdata(sig)[np.newaxis]
        binned = bin_obs(get_cds(sig4d), binning)

        if CFG['do_flat']:
            binned = correct_flat_field(cal['flat'], cal['dead'], binned[0][np.newaxis])

        return np.ma.getdata(binned[0]).astype(np.float32)

    dt_airs       = axis_info['AIRS-CH0-integration_time'].dropna().values.astype(np.float32)
    dt_airs[1::2] += 0.1
    dt_fgs1       = np.full(270000, 0.1, dtype=np.float32)
    dt_fgs1[1::2] += 0.1

    airs_raw = _process('AIRS-CH0', (32, 356), (32, 356), CFG['airs_bin'], dt_airs,
                        crop=(cut_inf, cut_sup))
    fgs1_raw = _process('FGS1',    (32,  32), (32,  32), CFG['fgs1_bin'], dt_fgs1)

    # Reorder AIRS to (time, wavelength, spatial)
    airs_cal = airs_raw.transpose(0, 2, 1)
    fgs1_cal = fgs1_raw

    return airs_cal, fgs1_cal

## Feature Engineering

In [ ]:
def _ts_stats(x):
    """Compact set of time-series summary statistics."""
    return np.array([
        np.nanmean(x), np.nanmedian(x), np.nanstd(x),
        skew(x), kurtosis(x),
        np.nanpercentile(x, 5),  np.nanpercentile(x, 10),
        np.nanpercentile(x, 25), np.nanpercentile(x, 75),
        np.nanpercentile(x, 90), np.nanpercentile(x, 95),
        np.nanmax(x) - np.nanmin(x),
    ], dtype=np.float32)


def _transit_feats(lc, out_n):
    """Transit geometry features: depth, duration, timing, asymmetry, SNR."""
    n       = len(lc)
    out_lc  = np.concatenate([lc[:out_n], lc[-out_n:]])
    in_lc   = lc[out_n:-out_n]
    baseline = np.nanmean(out_lc)

    depth   = 1.0 - np.nanmin(lc) / (baseline + 1e-10)
    below   = np.where(lc < baseline * 0.995)[0]
    ingress = float(below[0])  if len(below) else float(n // 2)
    egress  = float(below[-1]) if len(below) else float(n // 2)
    mid     = len(in_lc) // 2
    asym    = (np.nanmean(in_lc[:mid]) / (np.nanmean(in_lc[mid:]) + 1e-10)) if len(in_lc) > 2 else 1.0
    snr     = depth / (np.nanstd(out_lc) + 1e-10)

    x_out   = np.concatenate([np.arange(out_n), np.arange(n - out_n, n)])
    coeffs  = np.polyfit(x_out, out_lc, 2) if len(out_lc) > 3 else np.zeros(3)
    resid   = out_lc - np.polyval(coeffs, x_out)

    return np.array([
        depth,
        np.nanmean(in_lc) / (np.nanmean(out_lc) + 1e-10),
        ingress, egress, egress - ingress,
        asym, snr, coeffs[-2],          # slope of baseline linear term
        np.nanstd(resid), np.nanmax(np.abs(resid)),
        np.nanmean(resid), np.nanpercentile(resid, 5), np.nanpercentile(resid, 95),
    ], dtype=np.float32)


def extract_features(airs_cal, fgs1_cal, planet_id):
    """
    Build a (283, n_features) feature matrix for one planet.

    Layout: one row per wavelength channel (0-281 = AIRS, 282 = FGS1).
    AIRS channels are FGS1-detrended before feature computation.
    """
    n_time = airs_cal.shape[0]
    n_wl   = airs_cal.shape[1]
    out_n  = max(1, int(n_time * CFG['out_transit_frac']))

    airs_lc = np.nansum(np.where(np.isfinite(airs_cal), airs_cal, np.nan), axis=2)
    fgs1_lc = np.nansum(np.where(np.isfinite(fgs1_cal), fgs1_cal, np.nan), axis=(1, 2))

    fgs1_safe = np.where(fgs1_lc == 0, 1e-10, fgs1_lc)
    airs_norm = airs_lc / fgs1_safe[:, np.newaxis]

    fgs1_norm = fgs1_lc / (np.median(fgs1_lc) + 1e-10)
    fgs1_global = np.concatenate([
        _ts_stats(fgs1_norm),
        _transit_feats(fgs1_norm, out_n),
    ])

    baselines = np.nanmean(
        np.concatenate([airs_norm[:out_n], airs_norm[-out_n:]], axis=0), axis=0
    )
    depths_all = 1.0 - np.nanmin(airs_norm, axis=0) / (baselines + 1e-10)

    rows = []
    for wl in range(n_wl):
        lc = airs_norm[:, wl]
        if np.any(~np.isfinite(lc)):
            lc = pd.Series(lc).ffill().bfill().values.astype(np.float32)

        nbr     = depths_all[max(0, wl-5):min(n_wl, wl+6)]
        context = np.array([float(wl), wl / n_wl, np.nanmean(nbr), np.nanstd(nbr)],
                           dtype=np.float32)

        feat = np.concatenate([
            lc.astype(np.float32),
            _ts_stats(lc),
            _transit_feats(lc, out_n),
            context,
            fgs1_global,
        ])
        rows.append(feat)

    fgs1_pad = np.pad(fgs1_norm[:n_time], (0, max(0, n_time - len(fgs1_norm)))).astype(np.float32)
    fgs1_ctx = np.array([float(n_wl), 1.0,
                         1.0 - np.nanmin(fgs1_norm) / (np.nanmean(fgs1_norm) + 1e-10), 0.0],
                        dtype=np.float32)
    rows.append(np.concatenate([
        fgs1_pad,
        _ts_stats(fgs1_norm),
        _transit_feats(fgs1_norm, out_n),
        fgs1_ctx,
        fgs1_global,
    ]))

    arr = np.stack(rows).astype(np.float32)
    df  = pd.DataFrame(arr, columns=[f'f{i}' for i in range(arr.shape[1])])
    df['planet_id']     = planet_id
    df['wavelength_idx'] = np.arange(N_WL)
    return df

## Build Feature Matrices

In [ ]:
def get_planet_ids(split):
    dirs = glob.glob(os.path.join(PATH, split, '*'))
    return sorted(int(os.path.basename(d)) for d in dirs if os.path.basename(d).isdigit())


def build_feature_matrix(split, planet_ids):
    """Stream calibration + feature extraction, one planet at a time."""
    frames, failed = [], []
    for pid in tqdm(planet_ids, desc=split):
        try:
            airs, fgs1 = calibrate_planet(pid, split)
            frames.append(extract_features(airs, fgs1, pid))
            del airs, fgs1
            gc.collect()
        except Exception as e:
            print(f'  skip {pid}: {e}')
            failed.append(pid)
    if failed:
        print(f'Failed: {failed}')
    df = pd.concat(frames, ignore_index=True)
    print(f'{split}: {df.shape}')
    return df


train_ids = get_planet_ids('train')
test_ids  = get_planet_ids('test')
print(f'train={len(train_ids)}  test={len(test_ids)}')

In [ ]:
df_train = build_feature_matrix('train', train_ids)

In [ ]:
df_test = build_feature_matrix('test', test_ids)

In [ ]:
labels  = pd.read_csv(os.path.join(PATH, 'train_labels.csv'))
wl_cols = [c for c in labels.columns if c != 'planet_id']
assert len(wl_cols) == N_WL

labels_long = (labels
    .melt(id_vars='planet_id', value_vars=wl_cols, var_name='wl_col', value_name='target')
    .assign(wavelength_idx=lambda d: d['wl_col'].map({c: i for i, c in enumerate(wl_cols)}))
    [['planet_id', 'wavelength_idx', 'target']])

df_train = df_train.merge(labels_long, on=['planet_id', 'wavelength_idx'], how='left')
assert df_train['target'].isna().sum() == 0
print(f'train: {df_train.shape}  target [{df_train.target.min():.5f}, {df_train.target.max():.5f}]')

In [ ]:
feat_cols = [c for c in df_train.columns if c.startswith('f')]

train_X          = np.nan_to_num(df_train[feat_cols].values.astype(np.float32))
train_y          = df_train['target'].values.astype(np.float32)
train_planet_ids = df_train['planet_id'].values
train_wl_idx     = df_train['wavelength_idx'].values

test_X           = np.nan_to_num(df_test[feat_cols].values.astype(np.float32))
test_planet_ids  = df_test['planet_id'].values
test_wl_idx      = df_test['wavelength_idx'].values

print(f'train_X {train_X.shape}  test_X {test_X.shape}  features {len(feat_cols)}')

## Metric

In [ ]:
def gll_score(y_true, mu, sigma):
    """Gaussian log-likelihood (competition metric, higher = better)."""
    sigma = np.clip(sigma, CFG['sigma_floor'], None)
    return float(np.mean(-0.5 * (np.log(2 * np.pi * sigma**2) + ((y_true - mu) / sigma)**2)))


def per_wavelength_gll(y_true, mu, sigma, wl_idx):
    return {int(wl): gll_score(y_true[wl_idx==wl], mu[wl_idx==wl], sigma[wl_idx==wl])
            for wl in np.unique(wl_idx)}

## Cross-Validation

In [ ]:
def make_planet_folds(planet_ids_all, n_folds=5, seed=42):
    """KFold at planet level — prevents wavelength-row leakage across folds."""
    unique = np.unique(planet_ids_all)
    fold_map = np.zeros(len(unique), dtype=int)
    for fold, (_, vi) in enumerate(KFold(n_folds, shuffle=True, random_state=seed).split(unique)):
        fold_map[vi] = fold
    pid_to_fold = dict(zip(unique, fold_map))
    return np.array([pid_to_fold[p] for p in planet_ids_all])


fold_assignments = make_planet_folds(train_planet_ids, CFG['n_folds'], CFG['cv_seed'])
print('fold sizes:', np.bincount(fold_assignments))

In [ ]:
n_train = len(train_y)
n_test  = len(test_planet_ids)
n_folds = CFG['n_folds']

oof_mean = np.zeros(n_train, dtype=np.float32)
oof_q16  = np.zeros(n_train, dtype=np.float32)
oof_q84  = np.zeros(n_train, dtype=np.float32)
oof_et   = np.zeros(n_train, dtype=np.float32)

test_mean_folds = np.zeros((n_folds, n_test), dtype=np.float32)
test_q16_folds  = np.zeros((n_folds, n_test), dtype=np.float32)
test_q84_folds  = np.zeros((n_folds, n_test), dtype=np.float32)
test_et_folds   = np.zeros((n_folds, n_test), dtype=np.float32)

lgbm_models  = []
et_models    = []
fold_scores  = []

params_mean = CFG['lgbm_params']
params_q16  = {**CFG['lgbm_params'], **CFG['lgbm_q_extra'], 'alpha': CFG['q_lo']}
params_q84  = {**CFG['lgbm_params'], **CFG['lgbm_q_extra'], 'alpha': CFG['q_hi']}

cb = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)]

for fold in range(n_folds):
    tr  = fold_assignments != fold
    val = fold_assignments == fold

    Xtr, ytr = train_X[tr],  train_y[tr]
    Xv,  yv  = train_X[val], train_y[val]

    m = lgb.LGBMRegressor(**params_mean)
    m.fit(Xtr, ytr, eval_set=[(Xv, yv)], callbacks=cb)
    oof_mean[val]          = m.predict(Xv)
    test_mean_folds[fold]  = m.predict(test_X)
    lgbm_models.append(m)

    mq16 = lgb.LGBMRegressor(**params_q16)
    mq16.fit(Xtr, ytr, eval_set=[(Xv, yv)], callbacks=cb)
    oof_q16[val]          = mq16.predict(Xv)
    test_q16_folds[fold]  = mq16.predict(test_X)

    mq84 = lgb.LGBMRegressor(**params_q84)
    mq84.fit(Xtr, ytr, eval_set=[(Xv, yv)], callbacks=cb)
    oof_q84[val]          = mq84.predict(Xv)
    test_q84_folds[fold]  = mq84.predict(test_X)

    et = ExtraTreesRegressor(**CFG['et_params'])
    et.fit(Xtr, ytr)
    oof_et[val]          = et.predict(Xv)
    test_et_folds[fold]  = et.predict(test_X)
    et_models.append(et)

    raw_sig = np.abs(oof_q84[val] - oof_q16[val]) / 2.0 + 1e-8
    gll_lgbm = gll_score(yv, oof_mean[val], raw_sig)
    gll_et   = gll_score(yv, oof_et[val],   raw_sig)
    rmse     = np.sqrt(np.mean((yv - oof_mean[val])**2))
    fold_scores.append((gll_lgbm, gll_et))
    print(f'fold {fold+1}  rmse={rmse:.6f}  gll_lgbm={gll_lgbm:.4f}  gll_et={gll_et:.4f}')

    del mq16, mq84, et, Xtr, ytr, Xv, yv
    gc.collect()

lgbm_glls, et_glls = zip(*fold_scores)
print(f'\ncv lgbm gll: {np.mean(lgbm_glls):.4f} ± {np.std(lgbm_glls):.4f}')
print(f'cv et   gll: {np.mean(et_glls):.4f} ± {np.std(et_glls):.4f}')

## Ensemble

In [ ]:
test_mean      = test_mean_folds.mean(axis=0)
test_et        = test_et_folds.mean(axis=0)
test_sigma_ens = test_mean_folds.std(axis=0)

test_sigma_q = np.abs(test_q84_folds.mean(axis=0) - test_q16_folds.mean(axis=0)) / 2.0
oof_sigma_q  = np.abs(oof_q84 - oof_q16) / 2.0

_sig = np.clip(oof_sigma_q, CFG['sigma_floor'], None)

def neg_gll_blend(log_w):
    w = softmax(log_w)
    return -gll_score(train_y, w[0]*oof_mean + w[1]*oof_et, _sig)

res = minimize(neg_gll_blend, [1.0, 0.3], method='Nelder-Mead',
               options={'maxiter': 500, 'xatol': 1e-6})
w = softmax(res.x)
print(f'blend weights  lgbm={w[0]:.4f}  et={w[1]:.4f}')

w_lgbm, w_et = w
oof_blend    = w_lgbm * oof_mean  + w_et * oof_et
test_blend   = w_lgbm * test_mean + w_et * test_et

print(f'oof blend  rmse={np.sqrt(np.mean((train_y - oof_blend)**2)):.6f}')
print(f'oof blend  gll(raw sigma)={gll_score(train_y, oof_blend, _sig):.4f}')

## Uncertainty Calibration

In [ ]:
oof_sigma_comb  = oof_sigma_q
test_sigma_comb = np.sqrt(test_sigma_q**2 + test_sigma_ens**2)

# Fit per-wavelength scale factor k so that sigma_calibrated ≈ RMSE at each wavelength
calib_k = np.ones(N_WL)
for wl in range(N_WL):
    m  = train_wl_idx == wl
    mu = np.mean(oof_sigma_comb[m])
    if m.sum() >= 2 and mu > 1e-10:
        calib_k[wl] = np.std(train_y[m] - oof_blend[m]) / mu

print(f'calib_k  min={calib_k.min():.4f}  med={np.median(calib_k):.4f}  max={calib_k.max():.4f}')

def _apply_calib(sigma, wl_idx):
    out = sigma.copy()
    for wl in range(N_WL):
        out[wl_idx == wl] *= calib_k[wl]
    return np.maximum(out, CFG['sigma_floor'])

test_sigma = _apply_calib(test_sigma_comb, test_wl_idx)
oof_sigma  = _apply_calib(oof_sigma_comb,  train_wl_idx)

final_gll = gll_score(train_y, oof_blend, oof_sigma)
print(f'\nfinal oof gll: {final_gll:.4f}')

In [ ]:
wl_glls    = per_wavelength_gll(train_y, oof_blend, oof_sigma, train_wl_idx)
wl_gll_arr = np.array([wl_glls[wl] for wl in range(N_WL)])

print('worst wavelengths:')
for wl in np.argsort(wl_gll_arr)[:5]:
    print(f'  wl={wl:3d}  gll={wl_gll_arr[wl]:.4f}')
print('best wavelengths:')
for wl in np.argsort(wl_gll_arr)[-5:][::-1]:
    print(f'  wl={wl:3d}  gll={wl_gll_arr[wl]:.4f}')

## Spectral Smoothing (optional)

In [ ]:
if CFG['smooth_predictions']:
    def _smooth_spectra(pred, planet_ids):
        out = pred.copy()
        for pid in np.unique(planet_ids):
            m = planet_ids == pid
            if m.sum() == N_WL:
                out[m] = savgol_filter(pred[m], CFG['smooth_window'], CFG['smooth_poly'])
        return out

    oof_smooth  = _smooth_spectra(oof_blend,  train_planet_ids)
    gll_smooth  = gll_score(train_y, oof_smooth, oof_sigma)
    print(f'gll smooth={gll_smooth:.4f}  before={final_gll:.4f}')
    if gll_smooth > final_gll:
        test_blend = _smooth_spectra(test_blend, test_planet_ids)
        print('using smoothed predictions')
else:
    print('smoothing disabled')

## Submission

In [ ]:
sample_sub = pd.read_csv(os.path.join(PATH, 'sample_submission.csv'))
non_id     = [c for c in sample_sub.columns if c != 'planet_id']
n_half     = len(non_id) // 2
mu_cols    = non_id[:n_half]
sig_cols   = non_id[n_half:]
print(f'sample_sub {sample_sub.shape}  mu_cols={len(mu_cols)}  sig_cols={len(sig_cols)}')

In [ ]:
out = df_test[['planet_id', 'wavelength_idx']].copy()
out['mu']  = test_blend
out['sig'] = test_sigma

mu_wide  = out.pivot('planet_id', 'wavelength_idx', 'mu')
sig_wide = out.pivot('planet_id', 'wavelength_idx', 'sig')

mu_wide.columns  = mu_cols
sig_wide.columns = sig_cols

submission = pd.concat([mu_wide, sig_wide], axis=1).reset_index()[sample_sub.columns]

assert submission.shape == sample_sub.shape,  'shape mismatch'
assert not submission.isnull().any().any(),    'NaN in submission'
assert (submission[sig_cols].values > 0).all(), 'non-positive sigma'

submission.to_csv(os.path.join(OUT, 'submission.csv'), index=False)
print(f'saved  shape={submission.shape}  mu_mean={submission[mu_cols].values.mean():.6f}  sig_mean={submission[sig_cols].values.mean():.6f}')

## Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].scatter(train_y[:5000], oof_blend[:5000], alpha=0.2, s=2)
axes[0].plot([train_y.min(), train_y.max()], [train_y.min(), train_y.max()], 'r--')
axes[0].set(xlabel='True', ylabel='Predicted', title='OOF: Predicted vs True')

axes[1].plot(wl_gll_arr)
axes[1].axhline(final_gll, color='r', ls='--', label=f'mean={final_gll:.3f}')
axes[1].set(xlabel='Wavelength', ylabel='GLL', title='Per-wavelength OOF GLL')
axes[1].legend()

axes[2].hist(test_sigma, bins=60, edgecolor='k')
axes[2].set(xlabel='σ', ylabel='Count', title='Test sigma distribution')

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'diagnostics.png'), dpi=120)
plt.show()

print(f'train={len(train_ids)}  test={len(test_ids)}  features={len(feat_cols)}')
print(f'lgbm={w_lgbm:.4f}  et={w_et:.4f}  oof_gll={final_gll:.4f}')